# LightRAG: Simple and Fast Graph-Based Retrieval-Augmented Generation

## Overview

**LightRAG** is a graph-based RAG system designed to be *simpler, cheaper, and incrementally updatable* than heavier knowledge-graph approaches such as Microsoft GraphRAG. It was introduced by Guo et al. (HKU, 2024) in the paper *"LightRAG: Simple and Fast Retrieval-Augmented Generation"*.

Instead of treating documents as a flat pile of independent chunks (as in classic vector RAG), LightRAG uses an LLM to extract a **knowledge graph** of entities and relationships from the corpus, and then retrieves with a **dual-level** strategy that mixes precise entity lookups with broad, theme-level relationship lookups.

## Motivation

Classic vector RAG retrieves the top-k most similar chunks. This works well for "look up a fact" questions but struggles when the answer requires **connecting information scattered across the corpus** (multi-hop questions, "how does X relate to Y?", "what are the overarching causes of Z?"). The chunks are isolated, so the relational structure of the knowledge is lost.

Graph-based RAG fixes this by modelling entities and their relationships explicitly. Microsoft GraphRAG does this but relies on expensive **community detection** and **full re-indexing**. LightRAG keeps the graph idea but makes it lighter:

- **Dual-level retrieval** instead of community summaries.
- **Incremental updates**: new documents are merged into the existing graph without rebuilding everything.

## Key components

1. **Graph-based indexing** — an LLM extracts entities (nodes, with descriptions) and relationships (edges, with descriptions + keywords) from each chunk; duplicates are merged across chunks.
2. **Key–value profiling** — each entity/relation gets a text "profile" that is embedded into a vector index, so we can match the graph semantically.
3. **Dual-level retrieval** — the query is split into *low-level* keywords (specific entities/details) and *high-level* keywords (broad themes). Low-level keywords retrieve **entities** + their direct relations; high-level keywords retrieve **relations** + their endpoint entities.
4. **Context assembly + generation** — retrieved entities, relations, and their source chunks are combined into a single context the LLM uses to answer.

## Benefits

- Better on multi-hop / global questions than flat vector RAG.
- Cheaper and faster to update than community-based GraphRAG.
- Transparent: you can inspect exactly which entities, relations, and passages produced an answer.

<div style="text-align: center;">

<img src="../images/lightrag.png" alt="LightRAG dual-level retrieval over a knowledge graph" style="width:100%; height:100%;">
</div>

# Package Installation and Imports

The cell below installs all packages required to run this notebook.

In [2]:
# Install required packages
!pip install langchain langchain-openai langchain-community faiss-cpu networkx matplotlib pydantic python-dotenv pypdf tqdm


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

fatal: destination path 'RAG_TECHNIQUES' already exists and is not an empty directory.


### Environment setup

This notebook works with **either the OpenAI API or Azure OpenAI**. Create a `.env` file in the project root and fill in the credentials for whichever backend you use. The cell below shows the variables it expects; the setup cell that follows reads them with `load_dotenv()` and builds the matching client (toggle the `AZURE` flag there).

In [4]:
# Contents of your .env file (create it in the project root).
# Fill in OPENAI_API_KEY for plain OpenAI, OR the AZURE_* block to use Azure OpenAI.
AZURE_OPENAI_API_KEY=""
AZURE_OPENAI_ENDPOINT=""
AZURE_OPENAI_DEPLOYMENT_NAME=""                          # your Azure chat deployment name
TEXT_EMBEDDING_3_LARGE_DEPLOYMENT_NAME=""    # your Azure embeddings deployment name
AZURE_OPENAI_API_VERSION="2024-06-01"

OPENAI_API_KEY=""

### Imports and client setup

We use the same building blocks as the other notebooks in this repo: `ChatOpenAI` for the LLM, `OpenAIEmbeddings` for vectors, `FAISS` as the vector store, and `networkx` for the in-memory knowledge graph. Create a `.env` file with your `OPENAI_API_KEY` before running.

In [5]:
import os
from typing import List
from collections import defaultdict

import networkx as nx
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tqdm import tqdm
from pydantic import BaseModel, Field

from langchain_openai import (
    ChatOpenAI, OpenAIEmbeddings,
    AzureChatOpenAI, AzureOpenAIEmbeddings,
)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

# Choose the backend. Azure reads the AZURE_* variables; plain OpenAI reads OPENAI_API_KEY.
AZURE = True  # set to True to use Azure OpenAI

if AZURE:
    llm = AzureChatOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
        temperature=0,
    )
    embeddings = AzureOpenAIEmbeddings(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
        azure_deployment=os.getenv("TEXT_EMBEDDING_3_LARGE_DEPLOYMENT_NAME"),
    )
else:
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
    # A cheap, fast model is plenty for extraction + answering in this demo.
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

/var/folders/_s/h_zz770n1x5d7gd45mjsr4140000gn/T/ipykernel_50435/1707331979.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Load and chunk the document

We reuse the climate-change PDF that ships with the repo. We split it into overlapping chunks and keep a small subset so the graph-extraction step (one LLM call per chunk) stays fast and cheap for the demo. Each chunk gets a `chunk_id` so we can trace retrieved facts back to their source passage.

In [6]:
path = "../data/Understanding_Climate_Change.pdf"
documents = PyPDFLoader(path).load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

# Keep a small subset for a fast/cheap demo. Increase or remove this for the full corpus.
chunks = chunks[:20]
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"{len(chunks)} chunks ready for graph extraction")

20 chunks ready for graph extraction


## Step 1 — Define the extraction schema

LightRAG's index is a knowledge graph. We ask the LLM to return **structured** output (via Pydantic + `with_structured_output`) so every chunk yields a clean list of entities and relationships.

- **Entity** = a node: a canonical `name`, a `type`, and a short `description` grounded in the text.
- **Relationship** = an edge between two entities, with a `description` of *how* they relate and `keywords` summarising the relation at a high level (these keywords are what makes the *high-level* retrieval layer possible).

In [7]:
class Entity(BaseModel):
    name: str = Field(description="Canonical name of the entity, capitalized so it can be merged across chunks")
    type: str = Field(description="Entity type, e.g. concept, organization, person, location, event")
    description: str = Field(description="Concise description of the entity based ONLY on the text")

class Relationship(BaseModel):
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    description: str = Field(description="How the two entities are related, based on the text")
    keywords: str = Field(description="High-level keywords summarizing the nature of the relationship")

class GraphExtraction(BaseModel):
    entities: List[Entity] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)

### The extraction chain

The prompt instructs the model to extract only what is present in the text and to keep entity names consistent, which is what lets us **merge** the same entity when it appears in different chunks.

In [8]:
EXTRACTION_PROMPT = """You are an information-extraction system building a knowledge graph.

From the text below extract:
- entities: the key named things/concepts. For each give a name, a type, and a short description grounded in the text.
- relationships: meaningful connections between two extracted entities. For each give source, target, a description of the relation, and high-level keywords.

Rules:
- Only use information present in the text.
- Keep entity names consistent and capitalized so the same entity can be merged across chunks.
- A relationship's source and target must be entities you also list in `entities`.

Text:
\"\"\"{text}\"\"\""""

extraction_chain = ChatPromptTemplate.from_template(EXTRACTION_PROMPT) | llm.with_structured_output(GraphExtraction)

def extract_graph_from_chunk(text: str) -> GraphExtraction:
    """Run the LLM extraction on a single chunk of text."""
    return extraction_chain.invoke({"text": text})

## Step 2 — Build the knowledge graph (with deduplication)

We loop over the chunks and fold each extraction into a single `networkx` graph. The important part is the **merge logic**:

- If an entity (matched by its normalized name) already exists, we *append* the new description instead of creating a duplicate node.
- If a relationship between two entities already exists, we merge descriptions/keywords and record the extra source chunk.

We also keep `entity_chunks`: a map from each entity to the set of chunk IDs it appeared in. This lets retrieval pull back the **original source passages**, not just graph facts.

In [9]:
graph = nx.Graph()
entity_chunks = defaultdict(set)  # entity key -> set of chunk_ids where it appears

def norm(name: str) -> str:
    """Normalize an entity name so the same entity merges across chunks."""
    return name.strip().lower()

for chunk in tqdm(chunks, desc="Extracting graph"):
    cid = chunk.metadata["chunk_id"]
    try:
        extraction = extract_graph_from_chunk(chunk.page_content)
    except Exception as e:
        print(f"  skipped chunk {cid}: {e}")
        continue

    # --- entities -> nodes (merge by normalized name) ---
    for ent in extraction.entities:
        key = norm(ent.name)
        if not key:
            continue
        if graph.has_node(key):
            graph.nodes[key]["descriptions"].append(ent.description)
        else:
            graph.add_node(key, name=ent.name, type=ent.type, descriptions=[ent.description])
        entity_chunks[key].add(cid)

    # --- relationships -> edges (merge by node pair) ---
    for rel in extraction.relationships:
        s, t = norm(rel.source), norm(rel.target)
        if not s or not t or s == t:
            continue
        for key, raw in [(s, rel.source), (t, rel.target)]:
            if not graph.has_node(key):  # endpoint not extracted as an entity -> add a stub
                graph.add_node(key, name=raw, type="unknown", descriptions=[])
            entity_chunks[key].add(cid)
        if graph.has_edge(s, t):
            graph.edges[s, t]["descriptions"].append(rel.description)
            graph.edges[s, t]["keywords"].append(rel.keywords)
            graph.edges[s, t]["chunk_ids"].add(cid)
        else:
            graph.add_edge(s, t, descriptions=[rel.description], keywords=[rel.keywords], chunk_ids={cid})

print(f"Knowledge graph: {graph.number_of_nodes()} entities, {graph.number_of_edges()} relationships")

Extracting graph:   0%|          | 0/20 [00:00<?, ?it/s]/Users/simonebitti/Documents/GitHub/RAG_Techniques/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GraphExtraction(entities=...tion, sea level rise')]), input_type=GraphExtraction])
  return self.__pydantic_serializer__.to_python(
Extracting graph:   5%|▌         | 1/20 [00:04<01:23,  4.42s/it]/Users/simonebitti/Documents/GitHub/RAG_Techniques/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GraphExtraction(entities=...s='fuel type, energy')]), input_type=GraphExtraction])
  return self.__pydantic_serializer__.to_python(
Extracting graph:  10%|█         | 2/20 [00:08<01:13

Knowledge graph: 318 entities, 344 relationships


### Collapse merged descriptions into a single profile

Each node/edge currently holds a *list* of descriptions (one per chunk it was seen in). We collapse them into one deduplicated string that will become the text we embed and show to the LLM.

In [10]:
for _, data in graph.nodes(data=True):
    data["description"] = " ".join(dict.fromkeys(data["descriptions"]))

for _, _, data in graph.edges(data=True):
    data["description"] = " ".join(dict.fromkeys(data["descriptions"]))
    data["keywords"] = ", ".join(dict.fromkeys(data["keywords"]))

print("Example entity:", list(graph.nodes(data=True))[0])

Example entity: ('climate change', {'name': 'Climate Change', 'type': 'concept', 'descriptions': ['Significant, long-term changes in the global climate.', 'Recent changes in climate discussed as being primarily driven by human activities.', 'An environmental issue to which oil combustion contributes through CO2 and other pollutants.', 'A phenomenon contributed to by logging and land-use changes in boreal forests and by agriculture.', 'A phenomenon that agriculture contributes to through certain emissions.', 'A phenomenon whose effects are already being felt around the world and are projected to intensify.', 'The broader driver altering the timing and length of seasons.', 'The driver linked to increased frequency and severity of extreme weather events.', 'A problem addressed through mitigation and adaptation.', 'The problem that research and innovation aim to combat.', 'The pressing challenge discussed as needing informed action, cooperation, innovation, and commitment.'], 'description'

## Step 3 — Build the dual-level vector indexes

This is the heart of LightRAG. We build **two** separate FAISS indexes:

- **Entity index** — one document per node, embedding a profile string `name (type): description`. This powers the **low-level** (specific, detail-oriented) retrieval.
- **Relation index** — one document per edge, embedding `source -- target | keywords | description`. This powers the **high-level** (thematic, global) retrieval.

Separating them is what lets a single query hit both *specific facts* and *broad themes* at the same time.

In [11]:
entity_docs = [
    Document(
        page_content=f"{data['name']} ({data['type']}): {data['description']}",
        metadata={"entity": node},
    )
    for node, data in graph.nodes(data=True)
]

relation_docs = [
    Document(
        page_content=(
            f"{graph.nodes[u]['name']} -- {graph.nodes[v]['name']} | "
            f"keywords: {data['keywords']} | {data['description']}"
        ),
        metadata={"source": u, "target": v},
    )
    for u, v, data in graph.edges(data=True)
]

entity_index = FAISS.from_documents(entity_docs, embeddings)
relation_index = FAISS.from_documents(relation_docs, embeddings)
print(f"Indexed {len(entity_docs)} entities and {len(relation_docs)} relations")

Indexed 318 entities and 344 relations


## Step 4 — Split the query into low-level and high-level keywords

Before retrieving, LightRAG asks the LLM to decompose the question into two keyword sets:

- **low_level**: specific entities / concrete details → used against the *entity* index.
- **high_level**: broad themes / overarching topics → used against the *relation* index.

This mirrors the dual structure of the index.

In [12]:
class QueryKeywords(BaseModel):
    low_level: List[str] = Field(description="Specific entities, names, or concrete details in the question")
    high_level: List[str] = Field(description="Broad themes or overarching topics the question is about")

KEYWORD_PROMPT = """Extract two kinds of keywords from the user question to drive a dual-level graph retrieval.
- low_level: specific entities / concrete details.
- high_level: broad themes / overarching topics.

Question: {question}"""

keyword_chain = ChatPromptTemplate.from_template(KEYWORD_PROMPT) | llm.with_structured_output(QueryKeywords)

def extract_query_keywords(question: str) -> QueryKeywords:
    return keyword_chain.invoke({"question": question})

## Step 5 — Dual-level retrieval

Now we combine everything:

1. **Low level**: search the entity index with the low-level keywords → seed entities. For each seed entity we also pull its **direct relations** (graph neighbours) and the **source chunks** it came from.
2. **High level**: search the relation index with the high-level keywords → seed relations. Each relation brings in its two **endpoint entities** and the source chunks the relation came from.
3. We assemble three context blocks — entities, relationships, and original passages — that get handed to the LLM.

Notice how the graph turns a couple of keyword hits into a *connected neighbourhood* of facts, which is exactly what flat vector RAG cannot do.

In [19]:
def lightrag_retrieve(question: str, k_entities: int = 4, k_relations: int = 4):
    kws = extract_query_keywords(question)
    low_query = ", ".join(kws.low_level) or question
    high_query = ", ".join(kws.high_level) or question

    low_level_entities = set()  # direct hits from the entity index (low-level)
    selected_entities = set()
    selected_edges = set()
    relevant_chunks = set()

    # --- LOW LEVEL: specific entities + their direct relations ---
    for hit in entity_index.similarity_search(low_query, k=k_entities):
        e = hit.metadata["entity"]
        if not graph.has_node(e):
            continue
        low_level_entities.add(e)
        selected_entities.add(e)
        relevant_chunks |= entity_chunks.get(e, set())
        for neighbour in graph.neighbors(e):
            selected_edges.add(tuple(sorted((e, neighbour))))

    # --- HIGH LEVEL: thematic relations + their endpoint entities ---
    for hit in relation_index.similarity_search(high_query, k=k_relations):
        s, t = hit.metadata["source"], hit.metadata["target"]
        if not graph.has_edge(s, t):
            continue
        selected_edges.add(tuple(sorted((s, t))))
        selected_entities.update([s, t])
        relevant_chunks |= graph.edges[s, t].get("chunk_ids", set())

    # --- assemble context blocks ---
    entity_block = "\n".join(
        f"- {graph.nodes[e]['name']} ({graph.nodes[e]['type']}): {graph.nodes[e]['description']}"
        for e in selected_entities if graph.has_node(e)
    )
    relation_block = "\n".join(
        f"- {graph.nodes[u]['name']} -- {graph.nodes[v]['name']}: {graph.edges[u, v]['description']}"
        for u, v in selected_edges if graph.has_edge(u, v)
    )
    chunk_block = "\n---\n".join(chunks[c].page_content for c in sorted(relevant_chunks))

    return {
        "keywords": kws,
        "entities": entity_block,
        "relations": relation_block,
        "chunks": chunk_block,
        # raw selection sets, handy for visualization / debugging
        "low_level_entities": low_level_entities,
        "selected_entities": selected_entities,
        "selected_edges": selected_edges,
    }

## Step 6 — Generate the answer

Finally we feed the three retrieved blocks to the LLM and ask it to answer using only that context.

In [20]:
ANSWER_PROMPT = """Answer the question using ONLY the context below. The context comes from a knowledge graph
(entities and their relationships) plus the original source passages.

### Entities
{entities}

### Relationships
{relations}

### Source passages
{chunks}

Question: {question}

Answer:"""

answer_chain = ChatPromptTemplate.from_template(ANSWER_PROMPT) | llm

def lightrag_answer(question: str, **kwargs):
    ctx = lightrag_retrieve(question, **kwargs)
    response = answer_chain.invoke({
        "entities": ctx["entities"],
        "relations": ctx["relations"],
        "chunks": ctx["chunks"],
        "question": question,
    })
    return response.content, ctx

## Usage example

We ask a question that benefits from connecting concepts (a good showcase for graph retrieval) and inspect the low/high-level keywords the system chose.

In [21]:
question = "How do greenhouse gas emissions contribute to climate change?"
answer, ctx = lightrag_answer(question)

print("Low-level keywords: ", ctx["keywords"].low_level)
print("High-level keywords:", ctx["keywords"].high_level)
print("\n--- Retrieved relationships ---\n", ctx["relations"][:800])
print("\n=== Answer ===\n", answer)

/Users/simonebitti/Documents/GitHub/RAG_Techniques/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryKeywords(low_level=[...man impact on climate']), input_type=QueryKeywords])
  return self.__pydantic_serializer__.to_python(


Low-level keywords:  ['greenhouse gas emissions', 'climate change']
High-level keywords: ['environmental science', 'atmospheric processes', 'global warming', 'human impact on climate']

--- Retrieved relationships ---
 - Climate Change -- Ice Core Samples: Ice core samples provide historical records used to understand past climate conditions and predict future trends related to climate change.
- Adaptation -- Climate Change: Addressing climate change requires adaptation. Adaptation involves making adjustments to minimize damage caused by climate change.
- Greenhouse Gas Emissions -- Solar Power: Solar power is described as a versatile and scalable solution for reducing carbon emissions.
- Climate Change -- Intergovernmental Panel On Climate Change: The IPCC has documented climate changes extensively.
- Agriculture -- Climate Change: Agriculture contributes to climate change.
- Greenhouse Gas Emissions -- Natural Gas: Natural gas extraction and use contribute to greenhouse gas emissions

## Comparison with basic RAG

| Aspect | Basic (vector) RAG | LightRAG |
|---|---|---|
| Index unit | Isolated text chunks | Entities + relationships (graph) + chunks |
| Retrieval | Single top-k vector search | Dual-level: entities (low) + relations (high) |
| Multi-hop / global questions | Weak — chunks are disconnected | Strong — follows graph relations |
| Indexing cost | Low (just embed chunks) | Higher (one LLM extraction per chunk) |
| Updates | Trivial (add chunk) | Incremental merge into the graph |
| vs. Microsoft GraphRAG | — | No expensive community detection; cheaper, incrementally updatable |

**When to prefer LightRAG:** corpora where relationships matter (technical docs, research, anything with many interlinked entities) and questions that ask you to synthesize across the corpus. **When basic RAG is enough:** simple fact lookup where one passage already contains the answer, or when you want the cheapest possible indexing.

## Comparison with other graph-based RAG

This repo has two other knowledge-graph techniques. All three model entities and relationships, but they differ in *how* the graph is built and queried:

| Aspect | [Graph RAG](graph_rag.ipynb) (from-scratch) | [Microsoft GraphRAG](Microsoft_GraphRag.ipynb) | LightRAG (this notebook) |
|---|---|---|---|
| Graph construction | Custom code: LLM + spaCy concept extraction over `networkx` | Official `graphrag` package via CLI (black box) | Custom code: LLM structured extraction over `networkx` |
| Retrieval strategy | Traverse the graph from query-relevant nodes, add vector context | Local **and** global search over LLM-generated community summaries | Dual-level: entity index (low-level) + relation index (high-level) |
| Community detection | No | Yes — Leiden clustering + per-community summaries | No |
| Indexing cost | Moderate (LLM calls per chunk) | High (entity extraction **plus** community summarization) | Moderate (one LLM extraction per chunk) |
| Incremental updates | Manual | Expensive full re-index | Cheap — merge new nodes/edges into the existing graph |
| Global / thematic questions | Limited | Strong (global search over communities) | Strong (high-level keywords matched against relations) |
| Setup & dependencies | `networkx`, `spaCy`, an LLM | The `graphrag` package + its config | `networkx`, `FAISS`, an LLM |

**How to read this:** Graph RAG is the simplest hand-rolled graph approach but has no notion of broad themes. Microsoft GraphRAG is the most powerful for *global* questions (it summarizes whole communities) but is the heaviest to build and to update. LightRAG sits in between: it recovers the global/thematic angle through its high-level relation retrieval, while staying cheap to build and **incrementally updatable** — its main selling point over Microsoft GraphRAG.

## Additional considerations

- **Extraction cost & latency.** Indexing makes one LLM call per chunk. For large corpora, batch the calls, cache results, and persist the graph (e.g. `pickle` the `networkx` graph and `FAISS.save_local` the indexes).
- **Entity resolution.** We merge entities by normalized name. Production systems add fuzzy/alias matching (e.g. "US" vs "United States") to avoid duplicate nodes.
- **Incremental updates.** To add a document, extract its chunks and run the same merge loop, then re-embed only the new/changed nodes and edges — no full rebuild needed.
- **Tuning.** `k_entities` / `k_relations` control breadth vs. precision; chunk size affects extraction quality. Larger graphs may need limits on neighbour expansion to keep context within the model window.
- **Faithfulness.** Because the LLM both extracts and answers, errors can compound. Keep the source passages in the context (as we do) so the model can ground its answer in raw text, not just graph summaries.

## References

- Guo, Z. et al. (2024). *LightRAG: Simple and Fast Retrieval-Augmented Generation.* arXiv:2410.05779.
- Official implementation: https://github.com/HKUDS/LightRAG

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--lightrag)